# Pen2STEP — Ablation Study
**GPU:** A100 40GB (Colab Pro)  
**Tahmini süre:** ~2–4 saat (50 örnek × 5 koşul)  
**Çalıştır:** Runtime → Run All → bekle

In [ ]:
# ─── 1. GPU + Drive ───────────────────────────────────────────────────
import torch, os, shutil, json

assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > A100'
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/Pen2STEP'
for d in [DRIVE, f'{DRIVE}/results', f'{DRIVE}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'  # model Drive'da kalır
print('Drive hazır →', DRIVE)

In [ ]:
# ─── 2. Bağımlılıklar (~2-3 dk) ──────────────────────────────────────
!pip install -q transformers==4.44.0 accelerate bitsandbytes
!pip install -q cadquery svglib reportlab scipy easyocr pillow
print('✓ Kurulum tamam')

In [ ]:
# ─── 3. Kaynak kodu Drive'dan kopyala ────────────────────────────────
# Önce: Windows'tan src/ klasörünü Drive/Pen2STEP/src/ altına yükle

os.makedirs('/content/project', exist_ok=True)
os.chdir('/content/project')

SRC = f'{DRIVE}/src'
if os.path.exists(SRC):
    shutil.copytree(SRC, '/content/project/src', dirs_exist_ok=True)
    open('/content/project/src/__init__.py', 'a').close()
    print('✓ src/ kopyalandı')
else:
    raise FileNotFoundError(
        'Drive/Pen2STEP/src/ bulunamadı.\n'
        'Windows Explorer → src/ klasörünü sürükle → Drive/Pen2STEP/src/'
    )

In [ ]:
# ─── 4. Veri seti (Drive'da varsa yükle, yoksa üret ~5 dk) ───────────
import sys; sys.path.insert(0, '/content/project')
os.chdir('/content/project')

LOCAL_DATA = '/content/project/data'
DRIVE_DATA = f'{DRIVE}/data'

if os.path.exists(f'{DRIVE_DATA}/manifest.jsonl'):
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    n = sum(1 for _ in open(f'{LOCAL_DATA}/manifest.jsonl'))
    print(f'✓ Veri seti Drive\'dan yüklendi — {n} örnek')
else:
    print('Veri seti üretiliyor (~5 dk)...')
    !cd /content/project && python -m src.generate_dataset --n 3000 --out data --seed 0
    shutil.copytree(LOCAL_DATA, DRIVE_DATA, dirs_exist_ok=True)
    print('✓ Drive\'a kaydedildi — bir daha üretilmeyecek')

In [ ]:
# ─── 5. ABLASYON ─────────────────────────────────────────────────────
# T4 GPU: ~3-4 saat (20 örnek × 5 koşul)
# A100 GPU: ~1-2 saat (20 örnek × 5 koşul)
# Her koşul bitince Drive'a yedeklenir. Kesilirse tekrar çalıştır → devam eder.

os.chdir('/content/project')
RESULTS = '/content/project/data/results'
os.makedirs(RESULTS, exist_ok=True)

CONDITIONS = ['raw_iso', 'repair_iso', 'multiview', 'full_ocr', 'full_gt']

# Tamamlananları atla (resume)
done = {f.replace('done_','').replace('.flag','')
        for f in os.listdir(RESULTS) if f.startswith('done_')}
todo = [c for c in CONDITIONS if c not in done]

if done: print(f'Daha önce tamamlanan: {list(done)}')
print(f'Çalışacak koşullar: {todo}\n')

import time
total_start = time.time()

for i, cond in enumerate(todo, 1):
    print(f'[{i}/{len(todo)}] {cond} başlıyor...')
    t0 = time.time()

    ret = os.system(
        f'cd /content/project && '
        f'python -m src.run_baselines --data data --conditions {cond} --limit 20'
    )

    elapsed = (time.time() - t0) / 60
    total   = (time.time() - total_start) / 60

    if ret == 0:
        shutil.copytree(RESULTS, f'{DRIVE}/results', dirs_exist_ok=True)
        open(f'{RESULTS}/done_{cond}.flag', 'w').close()
        print(f'  ✓ {cond} — {elapsed:.1f} dk  (toplam: {total:.1f} dk)\n')
    else:
        print(f'  ❌ {cond} hata verdi ({elapsed:.1f} dk)\n')

print(f'\n✅ Ablasyon tamamlandı — toplam {(time.time()-total_start)/60:.1f} dk')

In [ ]:
# ─── 6. Sonuçları birleştir ve göster ────────────────────────────────
os.chdir('/content/project')
!python -m src.merge_results --data data

# Markdown tablosu
md = 'data/results/ablation.md'
if os.path.exists(md):
    print('\n' + '='*60)
    print('ABLASYON TABLOSU — makaleye yapıştır')
    print('='*60)
    print(open(md).read())

# JSON çıktısı (sonuç özeti için)
js = 'data/results/ablation.json'
if os.path.exists(js):
    print('\n' + '='*60)
    print('JSON çıktısı — sonuç tabloları için')
    print('='*60)
    print(open(js).read())

# Drive yedek
shutil.copytree(RESULTS, f'{DRIVE}/results', dirs_exist_ok=True)
print(f'\n✓ Sonuçlar Drive\'a kaydedildi → {DRIVE}/results')

In [ ]:
# ─── 7. Dosyaları indir ───────────────────────────────────────────────
from google.colab import files
for f in ['data/results/ablation.json', 'data/results/ablation.md']:
    if os.path.exists(f):
        files.download(f)
        print('İndirildi:', f)